In [1]:
import ast
import numpy as np
from numpy import random
import itertools
from scipy import stats

In [92]:

# Additive


# N= 20, K= 1 
f201=open(r'F:\Prof. Azevedo\Simulation of NK Model\Data of 0915_100 landscape_100 RW and 100 AW_N =100 and K=10~20\ADD\NK_model_Exp919_100_8_ADD.txt')
lines_f201 = f201.readlines()


landscape_rw_201 = lines_f201[0][27:]
landscape_rw_201 = ast.literal_eval(landscape_rw_201)

random_walk_201 = lines_f201[1][19:]
random_walk_201 = ast.literal_eval(random_walk_201)

# landscape_aw_201 = lines_f201[4][27:]
# landscape_aw_201 = ast.literal_eval(landscape_aw_201)

# adaptive_walk_201 = lines_f201[5][21:]
# adaptive_walk_201 = ast.literal_eval(adaptive_walk_201) 



In [93]:
def compare_str(str1, str2):
    list1 = list(str1)
    list2 = list(str2)
    
    index_list = [] 
        
    for i in range(len(list1)):
        if list1[i] != list2[i]:
            index_list.append(i)
        else:
            pass
        
    return index_list


def get_mutated_genotype(genotype, mutate_site):
        
    geno_list = list(genotype) 
        
    mutate_list = geno_list
        
    for i in mutate_site:
        if geno_list[i]=='0':
            mutate_list[i] = '1'
        else:
            mutate_list[i] = '0'
            
    mutate_geno = ''.join(mutate_list)
            
    return mutate_geno



def plus_certain_element(list1, index_list):  
    m = 0    
    for i in index_list:
        m+=list1[i]     
    return m



total_e_effect_rw = []

total_two_site_e_effect_rw = []
total_three_site_e_effect_rw = []
total_four_site_e_effect_rw = []
total_five_site_e_effect_rw = []


for p in range(100):
    each_e_effect_rw = []
    each_two_site_e_effect_rw = []
    each_three_site_e_effect_rw = []
    each_four_site_e_effect_rw = []
    each_five_site_e_effect_rw = []
    
    for q in range(100):
#         print(p,q)
        landscape_subset=landscape_rw_201[p][q]

        s = random_walk_201[p][q]

        genotype = []
        for t in s:
            genotype.append(t[0])


        changed_index_list = []

        for i in range(len(genotype)-1):
            changed_index = compare_str(genotype[i], genotype[i+1])
            changed_index_list.extend(changed_index)
    

        diff_allele_index = sorted(changed_index_list)


        genotype_subset= []

        for i in genotype:
            sequ = ''
            for j in diff_allele_index:
                sequ+=i[j]
        
            genotype_subset.append(sequ)
    

        subset_change_index = []

        for i in range(len(genotype_subset)-1):
            subset_change_index.extend(compare_str(genotype_subset[i], genotype_subset[i+1]))

        ancestor = genotype_subset[0]
        ancestor_list = list(ancestor)
        ancestor_fitness = landscape_subset[ancestor]

        one_mutant = []

        for i in subset_change_index:
            start = ancestor_list.copy()
            if ancestor_list[i] =='0':
                start[i] = '1'
            else:
                start[i] = '0'
            
            one_mutant.append(''.join(start))
    

        one_mutant_fitness = []

        for i in one_mutant:
            one_mutant_fitness.append(landscape_subset[i])


        w = []
        for i in one_mutant_fitness:
            w.append(i - ancestor_fitness)
    

        pairwise_site = []  # will be used to store the pairwise substitution sites (combined sites corresponding to 
                                # substitution 12, 13, 14, ..., 34, 35 and 45)
        pairwise_site_index = []  # will be used to store the index combinations of the pairwise substitutions 
                                    # (i.e, 12, 13, 14, ..., 34, 35 and 45 in substitution 12, 13, 14, ..., and 45)
        
        for L in range(len(subset_change_index)+1):
            for subset in itertools.combinations(subset_change_index, L):
                pairwise_site.append(list(subset))
         
        for j in range(len(subset_change_index)+1):
            for subset in itertools.combinations(range(len(subset_change_index)), j):
                pairwise_site_index.append(list(subset))


        pairwise_genotype = []   # will be used to store the pairwise genotypes
        pairwise_genotype_fitness = [] # will be used to store the fitness of pairwise genotypes
        pairwise_w = [] # will be used to store the calculated w of the pairwise genotypes
        
        for i in pairwise_site: # i is the combined pairwise substitution sites
            constructed_genotype = get_mutated_genotype(ancestor, i)  # pairwise substitution genotype
            pairwise_genotype.append(constructed_genotype)
            pairwise_genotype_fitness.append(landscape_subset[constructed_genotype]) # fitness of pairwise genotype
            pairwise_w.append(landscape_subset[constructed_genotype]-ancestor_fitness)


        pairwise_e_effect = [] # will be used to store the calculated e of each pairwised genotypes
        
        two_site_e_effect = []
        three_site_e_effect = []
        four_site_e_effect = []
        five_site_e_effect = []
        
        
        for i in pairwise_w:
            index_i = pairwise_w.index(i) # get the index of i 
            corr_one_mutant_index = pairwise_site_index[index_i] # get the indexes corresponding one site mutates(i.e., 1
                                                                     # and 2 for W1 and W2, 3 and 4 for W3 and W4) 
            plus_one_mutant_fitness = plus_certain_element(w, corr_one_mutant_index) # get the multiply results
                                                                        # (i.e., W1*W2, W1*W3,...)
                
            e_effect = i - plus_one_mutant_fitness
            pairwise_e_effect.append(e_effect) # calculate e and store in self.pairwise_e_effect
    
            if len(corr_one_mutant_index) == 2:
                two_site_e_effect.append(e_effect)
            elif len(corr_one_mutant_index) == 3:
                three_site_e_effect.append(e_effect)
            elif len(corr_one_mutant_index) == 4:
                four_site_e_effect.append(e_effect)
            elif len(corr_one_mutant_index) == 5:
                five_site_e_effect.append(e_effect)
    
    
        each_e_effect_rw.append(pairwise_e_effect)
        each_two_site_e_effect_rw.append(two_site_e_effect)
        each_three_site_e_effect_rw.append(three_site_e_effect)
        each_four_site_e_effect_rw.append(four_site_e_effect)
        each_five_site_e_effect_rw.append(five_site_e_effect)
        
    total_e_effect_rw.append(each_e_effect_rw)
    total_two_site_e_effect_rw.append(each_two_site_e_effect_rw)
    total_three_site_e_effect_rw.append(each_three_site_e_effect_rw)
    total_four_site_e_effect_rw.append(each_four_site_e_effect_rw)
    total_five_site_e_effect_rw.append(each_five_site_e_effect_rw)
        
with open('E_effect_1008_RW_0922.txt', 'w') as o:
    o.write('TOTAL E EFFECT_RW: {}\n'. format(total_e_effect_rw))
    o.write('2 Site E EFFECT_RW: {}\n'. format(total_two_site_e_effect_rw))
    o.write('3 Site E EFFECT_RW: {}\n'. format(total_three_site_e_effect_rw))
    o.write('4 Site E EFFECT_RW: {}\n'. format(total_four_site_e_effect_rw))
    o.write('5 Site E EFFECT_RW: {}\n'. format(total_five_site_e_effect_rw))